# Lecture 15 — Normalizing Flows

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

> **Data: Mixed.** Sections 1–9 use **synthetic / analytic** data (functions we sample by hand, plus the `digits` dataset). Section 10 uses a **real computed** dataset — an **alanine dipeptide molecular-dynamics trajectory** loaded with `mdtraj`, from which we extract the $(\phi, \psi)$ backbone dihedral angles (the Ramachandran plot).

---

## What you'll learn today

By the end of this notebook you will be able to:

1. Explain in plain words what a "normalizing flow" is and why it might be useful.
2. **Derive the 1D change-of-variables formula** and verify it numerically on any function $f$ you pick.
3. **Derive the training loss** (negative log-likelihood) from the maximum-likelihood principle.
4. **Compute by hand** why a coupling layer's Jacobian is triangular and its determinant is cheap.
5. Build a small flow in JAX (≈30 lines of code) and train it on two-dimensional toy data.
6. Apply that same flow to **real handwritten digits**, and train a **Boltzmann generator** that samples the **alanine dipeptide Ramachandran distribution** from its energy alone.

> **Runtime tip.** Everything runs on the free Colab CPU in about 5 minutes. No GPU needed.

> **What you should already know.** From earlier lectures: how to train a small neural network in JAX with `flax.nnx`, what a probability density is, and what a Gaussian is. That's it. We don't assume you remember Jensen's inequality.

<!-- lecture15-visual:start:title-flow -->
<!-- lecture15-visual:end:title-flow -->


## 0. Setup

In [ ]:
# On Colab, uncomment the next line. mdtraj is only needed for the physics
# section (Section 10 — alanine dipeptide); everything before it is pure JAX.
# !pip install -q jax jaxlib optax flax matplotlib scipy mdtraj

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from functools import partial

jax.config.update("jax_enable_x64", False)
key = jr.PRNGKey(42)   # default seed for the whole notebook (reproducibility)

print(f"JAX backend: {jax.default_backend()}")
print(f"master PRNG key: {key}")


## 1. The problem in one sentence

> *Given a bunch of data points $x_1, x_2, \ldots, x_N$, learn how to make **new ones** that look just like them.*

That's what a **generative model** does. Concretely:

- We have photos of cats → we want to make new cat photos.
- We have molecular conformations from MD → we want to draw new ones cheaply.
- We have lattice gauge field configurations → we want a fast sampler.

Mathematically, we want a probability density $p_\theta(x)$ such that

$$
p_\theta(x) \;\approx\; p_{\text{data}}(x).
$$

Once we have it, two things become possible:
- **Sample**: draw a fresh $x \sim p_\theta$.
- **Evaluate**: ask "how plausible is this particular $x$?" by computing $p_\theta(x)$.

### Where flows sit among the big-three generative model families

| | **VAE** (L14) | **GAN** | **Normalizing Flow** (today) |
|---|---|---|---|
| Sampling | one forward pass through decoder | one forward pass through generator | one forward pass through $f$ |
| Density $p_\theta(x)$ | **lower bound** only (ELBO) | not available at all | **exact**, in closed form |
| Training signal | reconstruction + KL | adversarial — can be unstable | maximum likelihood — a single scalar loss |
| Latent dim | usually < data dim (bottleneck) | flexible | **= data dim** (invertible) |
| Image quality at scale | blurry-ish | sharp | competitive but not SOTA |
| Killer use-case | learned compression, structured latents | photoreal generation | exact-density tasks: physics sampling, likelihood-based inference, anomaly scoring |

Today we'll see the **rightmost column**: a model that gives **exact** densities, not approximations, and produces independent samples in a single forward pass. The price we pay is that the latent must have the same dimension as the data — flows can't bottleneck.


## 2. The big idea — deform a Gaussian

Here's the central trick. We **already know** how to sample a complicated distribution: the Gaussian.

```
z ~ N(0, I)        ← we can do this in one line of code
```

Suppose we have a smooth, **invertible** function $f$ that maps the Gaussian world to the data world:

$$
x \;=\; f(z), \qquad z \sim \mathcal{N}(0, I).
$$

Then sampling $x$ is just: draw $z$, push it through $f$. **One forward pass. Independent samples. No Markov chain.**

The whole game is: **learn $f$**.

Think of it as squishing and stretching a blob of clay. The Gaussian is a perfectly round blob. The data is some weird shape. A flow learns how to deform one into the other.

### One $f$ is hard; a stack of small $f$'s is easy

In practice we never try to learn one big complicated $f$ in one shot. We build $f$ as a **composition** of many simple invertible steps:

$$
f \;=\; f_K \circ f_{K-1} \circ \cdots \circ f_2 \circ f_1.
$$

Each $f_k$ does a small, easy-to-invert deformation. Composing them gives an arbitrarily expressive overall map. This is **the** picture you'll see in every paper on normalizing flows — let's draw it.


In [ ]:
# Canonical "chain of bijections" picture: a sequence of nodes z_0, z_1, ..., z_K=x
# with arrows f_k between them, and the density getting progressively more complex.
fig = plt.figure(figsize=(13, 3.6))
gs = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.4], hspace=0.05)

# Top row: nodes and arrows representing the chain.
ax_top = fig.add_subplot(gs[0, :]); ax_top.axis("off")
ax_top.set_xlim(0, 10); ax_top.set_ylim(0, 1)

nodes_x = [0.6, 2.8, 5.0, 7.2, 9.4]
labels  = [r"$z_0 \sim \mathcal{N}(0, I)$", r"$z_1$", r"$z_2$", r"$z_3$", r"$z_K = x$"]
colors  = ["#e57373", "#ffb74d", "#fff176", "#aed581", "#4db6ac"]

for cx, lbl, col in zip(nodes_x, labels, colors):
    ax_top.add_patch(plt.Circle((cx, 0.5), 0.32, color=col, ec="#37474f", lw=1.3, zorder=3))
    ax_top.text(cx, 0.5, lbl, ha="center", va="center", fontsize=11, zorder=4)

for i in range(4):
    x1, x2 = nodes_x[i] + 0.35, nodes_x[i+1] - 0.35
    ax_top.annotate("", xy=(x2, 0.5), xytext=(x1, 0.5),
                    arrowprops=dict(arrowstyle="-|>", color="#37474f", lw=1.4))
    ax_top.text((x1+x2)/2, 0.72, fr"$f_{i+1}$", ha="center", fontsize=11, color="#1565c0")

ax_top.text(0.6, 0.05, "simple prior", ha="center", fontsize=9, color="#555")
ax_top.text(9.4, 0.05, "complex data", ha="center", fontsize=9, color="#555")

# Bottom row: 5 density sketches showing the distribution morphing along the chain.
xs = np.linspace(-4, 4, 400)
densities = [
    np.exp(-xs**2 / 2),                                                            # Gaussian
    np.exp(-(xs - 0.4)**2 / 1.8),                                                  # shifted/widened
    np.exp(-(xs + 1.2)**2 / 0.6) + 0.6 * np.exp(-(xs - 0.8)**2 / 0.8),             # skewed bimodal
    0.7*np.exp(-(xs + 1.6)**2 / 0.4) + np.exp(-(xs - 1.4)**2 / 0.5),               # clearer two modes
    0.9*np.exp(-(xs + 2.0)**2 / 0.25) + np.exp(-(xs - 1.6)**2 / 0.3)
                                       + 0.4*np.exp(-(xs - 0.2)**2 / 0.15),       # multi-modal "data"
]
for k, (d, col) in enumerate(zip(densities, colors)):
    ax = fig.add_subplot(gs[1, k])
    d_n = d / np.trapezoid(d, xs)
    ax.fill_between(xs, d_n, color=col, alpha=0.65); ax.plot(xs, d_n, color="#37474f", lw=1.1)
    ax.set_xlim(-4, 4); ax.set_ylim(0, max(d_n) * 1.15)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)

fig.suptitle("Normalizing flow = a chain of small invertible steps deforming a Gaussian into the data",
             fontsize=12, y=0.99)
plt.show()


**Read the picture.** Each $f_k$ is a *simple* invertible function (we'll meet specific choices later). The density (bottom row) doesn't change all at once — it grows complicated one small step at a time. **All the math from here on is about how to build each $f_k$ so that it's (i) easy to invert and (ii) gives us a cheap Jacobian.**


In [ ]:
# A picture is worth a paragraph. Let's watch a Gaussian get deformed.
key = jr.PRNGKey(0)
z = jr.normal(key, (2000, 2))  # 2000 samples from N(0, I)

# Three example deformations. We just hand-pick the functions for illustration.
def deform_A(z):  # mild stretch along x-axis
    return jnp.stack([z[:, 0] * 2.0, z[:, 1]], axis=1)

def deform_B(z):  # shear
    return jnp.stack([z[:, 0] + 0.5 * z[:, 1], z[:, 1]], axis=1)

def deform_C(z):  # nonlinear: bend into a "U"
    return jnp.stack([z[:, 0], z[:, 1] + 0.6 * z[:, 0]**2 - 1.0], axis=1)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
labels = ["Start: Gaussian", "Stretch $x$", "Shear", "Bend (nonlinear)"]
points = [z, deform_A(z), deform_B(z), deform_C(z)]

for ax, lbl, p in zip(axes, labels, points):
    p = np.asarray(p)
    ax.scatter(p[:, 0], p[:, 1], s=3, alpha=0.4, c="C0")
    ax.set_title(lbl); ax.set_aspect("equal")
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.grid(alpha=0.3)
plt.suptitle("Same 2000 Gaussian samples — pushed through different deformations $f$",
             fontsize=12)
plt.tight_layout(); plt.show()


**Take-away.** Even simple hand-picked deformations of a Gaussian already produce non-Gaussian shapes. A neural network gives us a **learnable** $f$ — we'll just choose its weights so that the output matches our data.

<!-- lecture15-visual:start:change-of-variables -->
<!-- lecture15-visual:end:change-of-variables -->


## 3. Warm-up in 1D — the "stretching factor"

We have a Gaussian source $z$ and a transformed variable $x = f(z)$. **What does the density of $x$ look like?**

Intuition: think of $z$ and $x$ as positions of grains of sand.

- Where $f$ **stretches** space (a big interval of $z$ gets mapped to a big interval of $x$), the sand gets **spread out** → density goes **down**.
- Where $f$ **squeezes** space (a big interval of $z$ collapses into a small interval of $x$), the sand gets **piled up** → density goes **up**.

The "amount of stretch" at a point is $\left|\dfrac{df}{dz}\right|$ — the local **slope**. Big slope means big stretch.

Conservation of probability gives the formula we'll use throughout:

$$
\boxed{\; p_X(x) \;=\; \frac{p_Z(z)}{\left|\dfrac{df}{dz}\right|}, \qquad x = f(z). \;}
$$

That's it. That's the entire mathematical content of normalizing flows in 1D. Let's see it in action.


In [ ]:
# Take a Gaussian z and stretch/squeeze it through f(z) = z + 0.6*sin(z).
# Then plot the density of x and compare with what the formula predicts.
N = 100_000
z = jr.normal(jr.PRNGKey(0), (N,))

def f(z):    return z + 0.6 * jnp.sin(z)
def fp(z):   return 1.0 + 0.6 * jnp.cos(z)   # df/dz, always positive -> f is invertible

x = f(z)

# Predicted density at any x: p_X(x) = p_Z(z) / |f'(z)| where z = f^{-1}(x).
# Easy version: parametrise by z, then x = f(z) and density follows.
z_grid = jnp.linspace(-4, 4, 1000)
x_grid = f(z_grid)
pz_grid = jnp.exp(-z_grid**2 / 2) / jnp.sqrt(2*jnp.pi)
px_predicted = pz_grid / jnp.abs(fp(z_grid))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(np.asarray(z), bins=80, density=True, color="C0", alpha=0.6, label="$z$ samples")
axes[0].plot(z_grid, pz_grid, "k-", lw=2, label=r"$\mathcal{N}(0,1)$")
axes[0].set_title("Source: Gaussian"); axes[0].legend(); axes[0].set_xlabel("$z$")

axes[1].hist(np.asarray(x), bins=80, density=True, color="C1", alpha=0.6, label="$x = f(z)$ samples")
axes[1].plot(x_grid, px_predicted, "k-", lw=2, label="predicted $p_X(x)$")
axes[1].set_title("After $f(z) = z + 0.6\\sin(z)$"); axes[1].legend(); axes[1].set_xlabel("$x$")

for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Watch carefully.** The histogram (orange) matches the predicted density (black line). The shape is bumpy because where $\cos z$ is small the slope is small, the space is squeezed, and probability piles up.

Now everything in this notebook is going to be a higher-dimensional version of this picture, and a learnable version of this $f$.


### A back-of-the-envelope verification

If the formula $p_X(x) = p_Z(z)/|f'(z)|$ is right, we should be able to **predict the histogram of $x$ from scratch**, without simulating anything. Let's do it once more, this time without sneaking a peek at the orange bars.

The recipe in three lines:

1. Pick a dense grid of $z$ values.
2. At each grid point compute $p_Z(z) = \tfrac{1}{\sqrt{2\pi}}\,e^{-z^2/2}$ and the slope $f'(z)$.
3. Plot the pair $\bigl(f(z),\; p_Z(z)/|f'(z)|\bigr)$. That's the predicted density curve.

Nothing in this recipe touched a single random sample. Yet…


In [ ]:
# Build the predicted density curve purely from the formula. No samples involved.
z_grid_fine = jnp.linspace(-6, 6, 2000)
x_table  = f(z_grid_fine)
pz_table = jnp.exp(-z_grid_fine**2 / 2) / jnp.sqrt(2*jnp.pi)
px_table = pz_table / jnp.abs(fp(z_grid_fine))

# Sort by x so we can draw a clean line.
order = jnp.argsort(x_table)
x_sorted, px_sorted = x_table[order], px_table[order]

# Now we plot the orange histogram (samples) and overlay the green prediction.
plt.figure(figsize=(7, 3.5))
plt.hist(np.asarray(x), bins=80, density=True, color="C1", alpha=0.45,
         label="histogram of $x = f(z)$  (made from samples)")
plt.plot(np.asarray(x_sorted), np.asarray(px_sorted), "g-", lw=2,
         label=r"prediction $p_Z(z)/|f'(z)|$  (formula only)")
plt.title("Change of variables in 1D — formula vs. histogram"); plt.xlabel("$x$")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


The green curve and the orange bars agree. Two completely different procedures — sampling, and reading off a formula — produced the same density. That's what *"probability is conserved under invertible transforms"* means in practice.

### Your turn — try your own $f$

Edit the cell below to pick a new $f$ and see the formula predict the new density. A few ideas to try first:

- $f(z) = z^3$ — stretches the tails, compresses the middle. Watch the peak shoot up.
- $f(z) = \tanh(z)$ — squashes everything into $(-1, 1)$. $f'(z) = 1 - \tanh^2(z)$ is tiny near $\pm 1$, so probability **piles up** at the edges.
- $f(z) = 2z + \sin(3z)$ — wavy but still monotone. Density gets a bumpy comb-like pattern.
- $f(z) = z + 1.5\sin(z)$ — $f'$ goes **negative** somewhere, so $f$ is no longer one-to-one. **This is a cautionary tale.** The single-branch change-of-variables formula
  $$ p_X(x) \;=\; \frac{p_Z\!\big(f^{-1}(x)\big)}{\big|f'\!\big(f^{-1}(x)\big)\big|} $$
  is *only exact when $f$ is one-to-one*. When $f$ folds, several $z$ values map to the **same** $x$, so the true density must **sum over every preimage branch**:
  $$ p_X(x) \;=\; \sum_{k:\,f(z_k)=x} \frac{p_Z(z_k)}{|f'(z_k)|}. $$
  In the cell below, the green curve uses only the *single-branch* formula evaluated along the $z$-grid, so **in the folded region the green curve is wrong** — it traces individual branches instead of their sum. The **orange histogram is correct**, because sampling automatically adds up the contributions of all branches that land on the same $x$. Watch the two disagree where $f$ turns over. (This is exactly why all the flows we build below insist that $f$ be **invertible** — folding is forbidden by construction.)

We let `jax.grad` compute $f'$ for us so you can drop in any function.


In [ ]:
# === EDIT THIS CELL ===
def f_user(z):
    return jnp.tanh(z)            # try z**3, 2*z + jnp.sin(3*z), z + 1.5*jnp.sin(z), ...

# Auto-differentiate to get f'.  vmap so it works on a whole array of z at once.
fprime_user = jax.vmap(jax.grad(lambda zi: f_user(zi)))

# Sample and predict, same as above.
N = 60_000
z_s = jr.normal(jr.PRNGKey(0), (N,))
x_s = f_user(z_s)

z_grid = jnp.linspace(-5, 5, 2000)
x_grid = f_user(z_grid)
pz_grid = jnp.exp(-z_grid**2 / 2) / jnp.sqrt(2*jnp.pi)

# === TODO 1 — implement the 1D change-of-variables formula ===================
# Fill in px_grid so it equals the predicted density  p_X(x) = p_Z(z) / |f'(z)|.
# Hints:
#   * p_Z(z) is already computed above as `pz_grid`.
#   * f'(z) is `fprime_user(z_grid)`.
#   * Why must you wrap f' in jnp.abs()?  (A reflection f' < 0 still moves the
#     same amount of probability — density can never be negative.)
#   * Add a tiny epsilon (e.g. + 1e-8) inside the |.| so you never divide by 0.
# NOTE: this is the *single-branch* formula. It is exact ONLY when f is one-to-one.
#       For a folding f (try f_user = z + 1.5*sin(z)) the single-branch formula
#       breaks here — the green curve will disagree with the orange histogram.
# px_grid = ...                      # <-- TODO
px_grid = pz_grid / (jnp.abs(fprime_user(z_grid)) + 1e-8)   # reference solution
# Checkpoint: for a monotone f (the default tanh), the green curve should land
# right on top of the orange histogram. If they overlap, your formula is correct.
# ============================================================================

order = jnp.argsort(x_grid)
plt.figure(figsize=(7, 3.5))
plt.hist(np.asarray(x_s), bins=80, density=True, color="C1", alpha=0.45, label="samples")
plt.plot(np.asarray(x_grid)[order], np.asarray(px_grid)[order], "g-", lw=2,
         label=r"formula $p_Z(z)/|f'(z)|$")
plt.title(r"Your $f$ vs the change-of-variables formula")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


**Pause and predict before each experiment.** Decide *where* you expect the density to be tall and short before running the cell. That habit of "imagine $f'$, then imagine where probability piles up" is exactly the intuition you'll lean on for every flow in this notebook.


## 3.5 Where does the training loss come from?

Later on we'll just write
```python
loss = -flow.log_prob(x).mean()
```
and minimise it. Where did that one-liner come from? Let's derive it once, with no hand-waving.

### From "best model" to a formula you can put in a loss

We have data $\{x_1, \ldots, x_N\}$ and a family of candidate densities $p_\theta(x)$ controlled by parameters $\theta$ (the flow's weights). We want the $\theta$ that **makes the observed data as plausible as possible**.

If we treat the data as independent draws, the joint probability of seeing this particular dataset is

$$
\mathcal{L}_{\text{joint}}(\theta) \;=\; \prod_{i=1}^{N} p_\theta(x_i).
$$

The **maximum likelihood estimator** is the $\theta$ that maximises this product. Two tweaks make it numerically friendly:

**Tweak 1 — take the log.** Products of many small probabilities underflow; sums of their logs don't.

$$
\theta^\star \;=\; \arg\max_\theta \;\; \sum_{i=1}^N \log p_\theta(x_i).
$$

**Tweak 2 — divide by $N$ and flip the sign.** Dividing by $N$ makes the loss independent of batch size; flipping the sign turns it into a *minimisation*, which is what every optimiser expects.

$$
\theta^\star \;=\; \arg\min_\theta \;\;\underbrace{- \frac{1}{N}\sum_{i=1}^N \log p_\theta(x_i)}_{\text{negative log-likelihood (NLL)}}.
$$

That's the entire derivation. In code:

```python
loss = -flow.log_prob(x).mean()       # mean over the batch
```

A literal translation. Nothing magical.


In [ ]:
# A tiny numerical example, using a 2-parameter family (Gaussian with mean mu, std sigma).
# This is the same procedure the flow will use, but with only two knobs.

data = jnp.array([-0.3, 0.1, 0.7, -1.2, 0.5])      # five toy points

def gaussian_log_prob(x, mu, sigma):
    return -0.5 * ((x - mu) / sigma)**2 - jnp.log(sigma) - 0.5 * jnp.log(2*jnp.pi)

candidates = [
    ("N(0, 1)",                                                     0.0, 1.0),
    ("N(1, 2)  -- too far off, too wide",                           1.0, 2.0),
    ("N(empirical mean, empirical std)  <-- the MLE",
     float(jnp.mean(data)), float(jnp.std(data))),
]

print(f"{'candidate model':<55s}  mean log p   ->   loss")
print("-" * 80)
for name, mu, sigma in candidates:
    mean_logp = float(gaussian_log_prob(data, mu, sigma).mean())
    print(f"{name:<55s}  {mean_logp:+.4f}     {-mean_logp:+.4f}")


**Read the numbers.** The MLE candidate (last row) has the **largest mean log-likelihood** → the **smallest NLL**. Picking $\theta$ to minimise the NLL is identical to picking $\theta$ to maximise the data's plausibility — by construction.

The only difference for a flow is that $p_\theta(x)$ is a much richer family than "Gaussian with two knobs". The optimiser still does the same job: nudge $\theta$ in the direction that lowers $-\overline{\log p}$.


## 4. Higher dimensions — and the one technical wrinkle

In $d$ dimensions the slope $|df/dz|$ becomes the **Jacobian determinant** $|\det J_f|$, which measures how a small volume element gets resized.

$$
p_X(x) \;=\; \frac{p_Z(z)}{|\det J_f(z)|}, \qquad x = f(z).
$$

This is the same formula you've used in physics whenever you change coordinates — for example $dx\,dy = r\,dr\,d\theta$, where the Jacobian factor is $r$.

For most general neural networks $f$, two things go wrong:

1. The network might **not be invertible** at all — we couldn't undo $x = f(z)$.
2. Computing $\det J_f$ costs $O(d^3)$. For an image with $d \sim 10^4$ pixels, that's billions of operations. Hopeless.

> The whole architecture design of normalizing flows is one long answer to: **how do we build an invertible network whose Jacobian determinant is cheap to compute?**

We'll see one famous answer next: **coupling layers** (Dinh, Sohl-Dickstein & Bengio 2016, "RealNVP").


## 5. The coupling-layer trick

The trick is almost embarrassingly simple. Split the dimensions of $z$ into two groups, say $z = (z_A, z_B)$. Then build a one-layer flow as:

$$
x_A = z_A \qquad\text{(leave half the dimensions alone)}
$$

$$
x_B = z_B \cdot e^{\,s(z_A)} \;+\; t(z_A) \qquad\text{(scale and shift the other half, based on the first half)}
$$

Here $s(\cdot)$ and $t(\cdot)$ are **any** neural networks. They produce a scaling factor $e^{s}$ and a shift $t$ for each transformed dimension.

### Why this works

**Invertibility.** Given $x_A, x_B$, recover the inputs by:

- $z_A = x_A$ (we kept it!)
- $z_B = (x_B - t(x_A)) \cdot e^{-s(x_A)}$

No matrix inverse, no iterative solver. We literally just kept half the input and stored enough information to undo the rest.

**Tractable Jacobian.** The Jacobian matrix of this map looks like

$$
J \;=\;
\begin{pmatrix}
I & 0 \\
\star & \mathrm{diag}\,e^{s(z_A)}
\end{pmatrix}
$$

which is **triangular**. The determinant of a triangular matrix is just the product of its diagonal:

$$
\det J \;=\; \prod_{i \in B} e^{\,s_i(z_A)} \;=\; e^{\sum_i s_i(z_A)}.
$$

So $\log|\det J|$ is just a sum — $O(d)$, not $O(d^3)$. Done.

### One layer isn't enough

A single coupling layer leaves half the dimensions untouched — so it's not very expressive. The fix: **stack many** coupling layers, and **alternate** which group is kept fixed:

- Layer 1: keep $A$ fixed, transform $B$.
- Layer 2: keep $B$ fixed, transform $A$.
- Layer 3: keep $A$ fixed, transform $B$.
- …

After half-a-dozen layers every dimension has been transformed conditioned on every other dimension, and the flow can model arbitrarily complicated shapes.

<!-- lecture15-visual:start:coupling-layer -->
<!-- lecture15-visual:end:coupling-layer -->


In [ ]:
# Cartoon diagram of one coupling layer.
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.6); ax.axis("off")

def box(x, y, w, h, label, fc, ec="#37474f"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                 fc=fc, ec=ec, lw=1.2))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

def arr(x1, y1, x2, y2, color="black"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.3))

box(0.2, 2.4, 1.0, 0.9, r"$z_A$", "#e3f2fd")          # input z_A (kept)
box(0.2, 0.8, 1.0, 0.9, r"$z_B$", "#fce4ec")          # input z_B (transformed)
box(3.0, 1.9, 1.6, 1.0, r"$s, t$" + "\n(neural net)", "#fff3e0")
box(5.8, 0.8, 2.2, 0.9, r"$z_B \cdot e^{s} + t$", "#fce4ec", ec="#c2185b")
box(8.8, 2.4, 1.0, 0.9, r"$x_A$", "#e3f2fd")
box(8.8, 0.8, 1.0, 0.9, r"$x_B$", "#fce4ec")

arr(1.2, 2.85, 3.0, 2.55)   # z_A -> net
arr(1.2, 2.85, 8.8, 2.85)   # z_A -> x_A (kept)
arr(4.6, 2.20, 5.8, 1.45)   # net -> scale&shift
arr(1.2, 1.25, 5.8, 1.25)   # z_B -> scale&shift
arr(8.0, 1.25, 8.8, 1.25)   # -> x_B

ax.text(5.0, 3.2, "kept identical", fontsize=10, color="#1565c0", ha="center")
ax.text(5.0, 0.25, r"reversible:  $z_B = (x_B - t)\,e^{-s}$",
        fontsize=11, ha="center", color="#c2185b")
plt.tight_layout(); plt.show()


## 5.5 Hand-computed Jacobian — making the "triangular" claim real

The trickiest line in the previous section is: *the Jacobian is triangular, so its determinant is just the product of the diagonal.* Let's stop trusting our intuition and **prove it on a concrete example** before we trust the framework.

We'll work in 4D and split as: dimensions $(0, 1)$ are **kept**, dimensions $(2, 3)$ are **transformed**. The map is

$$
\begin{aligned}
x_0 &= z_0, \\
x_1 &= z_1, \\
x_2 &= z_2 \cdot e^{s_0(z_0, z_1)} + t_0(z_0, z_1), \\
x_3 &= z_3 \cdot e^{s_1(z_0, z_1)} + t_1(z_0, z_1).
\end{aligned}
$$

We hand-pick $s$ and $t$ as readable little functions (no neural network) so you can sanity-check the algebra. Then we ask JAX to compute the 4×4 Jacobian and we look at it directly.


In [ ]:
# Concrete 4D coupling layer with hand-picked s, t.
def coupling_4d(z):
    z0, z1, z2, z3 = z[0], z[1], z[2], z[3]
    # s and t depend ONLY on the kept dims (z0, z1). That's the only rule.
    s = jnp.array([0.5 * z0 - 0.2 * z1,            # s_0 for x_2
                   0.3 * z0 + 0.4 * z1])           # s_1 for x_3
    t = jnp.array([0.1 * z0**2 + z1,               # t_0 for x_2
                   z0 - 0.6 * z1])                 # t_1 for x_3
    x = jnp.array([z0,                             # kept identically
                   z1,                             # kept identically
                   z2 * jnp.exp(s[0]) + t[0],      # scaled and shifted
                   z3 * jnp.exp(s[1]) + t[1]])
    return x, s

z_pt = jnp.array([0.3, -0.7, 1.1, -0.4])
x_pt, s_pt = coupling_4d(z_pt)
print(f"z = {z_pt}")
print(f"x = {x_pt}")
print(f"s at this point: s_0 = {float(s_pt[0]):+.4f},  s_1 = {float(s_pt[1]):+.4f}")
print()

# Ask JAX for the full 4x4 Jacobian numerically.
J = jax.jacobian(lambda z: coupling_4d(z)[0])(z_pt)
print("Jacobian J (row i = dx_i/dz):")
print(np.array_str(np.asarray(J), precision=3, suppress_small=True))


**Look at that matrix.** Three things should jump out:

| block | what it is | why |
|---|---|---|
| top-left 2×2 | identity | $x_0=z_0,\; x_1=z_1$ |
| top-right 2×2 | exactly zero | kept dims don't depend on transformed dims |
| bottom-left 2×2 | non-zero | the scale/shift used $z_0, z_1$ — but it doesn't matter for the determinant |
| bottom-right 2×2 | diagonal $\,e^{s_0}, e^{s_1}$ | because $\partial x_i / \partial z_i = e^{s_i}$ and $\partial x_2/\partial z_3 = 0$ |

So $J$ is **lower-triangular**. The determinant of a triangular matrix is the product of its diagonal:

$$
\det J \;=\; 1 \cdot 1 \cdot e^{s_0} \cdot e^{s_1} \;=\; e^{\,s_0 + s_1},
\qquad
\log|\det J| \;=\; s_0 + s_1.
$$

The bottom-left block can have absolutely anything in it (huge numbers, derivatives of crazy functions) — the determinant doesn't care. **That's the magic.**


In [ ]:
# Verify the determinant identity numerically.
det_from_J     = float(jnp.linalg.det(J))
det_from_s     = float(jnp.exp(s_pt.sum()))
logabsdet_J    = float(jnp.linalg.slogdet(J)[1])
logabsdet_form = float(s_pt.sum())

print(f"det J     (from jax.linalg)       = {det_from_J:.6f}")
print(f"exp(s_0 + s_1)                    = {det_from_s:.6f}")
print()
print(f"log|det J| (from jax)             = {logabsdet_J:.6f}")
print(f"s_0 + s_1                         = {logabsdet_form:.6f}")


The two pairs of numbers agree to floating-point precision. **This is why coupling layers scale.** A general $d \times d$ determinant costs $O(d^3)$. Ours costs $O(d)$ — just summing $d/2$ numbers. For $d = 10^4$ pixels of a small image that's the difference between *training overnight* and *not training, ever*.

> **Mini-exercise** (no submission, just for your own sake): pick a different split — keep dims $(0, 2)$ and transform $(1, 3)$. What does $J$ look like now? Is it still triangular? *(Hint: yes, after a row/column reordering.)*


## 6. Hands-on A — code one coupling layer

We're going to work in 2D ($d = 2$), so each group is a single coordinate. Each coupling layer is a small MLP $\to (s, t)$ followed by the scale-and-shift step.

Read the comments carefully — the code is the math.


In [ ]:
class AffineCoupling(nnx.Module):
    # mask[i] == 1  ->  dimension i is "kept"
    # mask[i] == 0  ->  dimension i is "transformed"
    def __init__(self, d, hidden, mask, *, rngs):
        self.mask = mask
        # The s and t networks are an ordinary MLP. They look at the kept dims
        # and produce a scaling and a shift for the transformed dims.
        self.net = nnx.Sequential(
            nnx.Linear(d, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, 2 * d, rngs=rngs),   # outputs s and t stacked
        )
        self.d = d

    def _scale_and_shift(self, kept):
        st = self.net(kept)
        s_raw, t = st[..., :self.d], st[..., self.d:]
        s = jnp.tanh(s_raw) * 2.0   # bound |s| < 2 for training stability
        return s, t

    def forward(self, z):
        # Input z, output x = (z_A, z_B * exp(s) + t).
        s, t = self._scale_and_shift(z * self.mask)
        transformed = 1.0 - self.mask
        x = z * jnp.exp(s * transformed) + t * transformed
        # === TODO 2 — log|det J| of this coupling layer ====================
        # From Section 5: the Jacobian of a coupling layer is triangular, so
        # its determinant is the product of the diagonal entries exp(s) on the
        # TRANSFORMED dimensions. Hence  det J = exp(s_0 + s_1 + ...)  and
        #     log|det J| = sum of s over the transformed dims.
        # The kept dims contribute exp(0) = 1, so multiply s by `transformed`
        # (a 0/1 mask) before summing, then sum over the last axis.
        # log_det = ...                                   # <-- TODO
        log_det = jnp.sum(s * transformed, axis=-1)        # reference solution
        # Checkpoint: after training (Section 7), the NLL should DECREASE each
        # epoch. A wrong/missing log_det makes the loss diverge or stay flat.
        # ===================================================================
        return x, log_det

    def inverse(self, x):
        # Input x, output z = (x_A, (x_B - t) * exp(-s)).
        s, t = self._scale_and_shift(x * self.mask)
        transformed = 1.0 - self.mask
        z = (x - t * transformed) * jnp.exp(-s * transformed)
        log_det = -jnp.sum(s * transformed, axis=-1)
        return z, log_det


class RealNVP(nnx.Module):
    # A stack of coupling layers with alternating masks.
    def __init__(self, d=2, n_layers=8, hidden=64, *, rngs):
        masks = []
        for i in range(n_layers):
            mask = jnp.zeros(d)
            mask = mask.at[:d//2].set(1.0) if i % 2 == 0 else mask.at[d//2:].set(1.0)
            masks.append(mask)
        self.layers = nnx.List([AffineCoupling(d, hidden, m, rngs=rngs) for m in masks])
        self.d = d

    def forward(self, z):
        # Push z through every layer, accumulating log|det J|.
        log_det = jnp.zeros(z.shape[0]); x = z
        for layer in self.layers:
            x, ld = layer.forward(x); log_det += ld
        return x, log_det

    def inverse(self, x):
        log_det = jnp.zeros(x.shape[0]); z = x
        for layer in reversed(self.layers):
            z, ld = layer.inverse(z); log_det += ld
        return z, log_det

    def log_prob(self, x):
        # log p(x) = log p_Z(z) + log|det J_{f^{-1}}(x)|, with z = f^{-1}(x).
        z, log_det_inv = self.inverse(x)
        log_pz = -0.5 * jnp.sum(z**2, axis=-1) - 0.5 * self.d * jnp.log(2 * jnp.pi)
        return log_pz + log_det_inv

    def sample(self, key, n):
        z = jr.normal(key, (n, self.d))
        x, _ = self.forward(z)
        return x


In [ ]:
# Quick sanity check: forward then inverse should recover the input.
flow = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(0))
z_test = jr.normal(jr.PRNGKey(1), (200, 2))
x_test, _ = flow.forward(z_test)
z_back, _ = flow.inverse(x_test)
print(f"Max round-trip error: {float(jnp.max(jnp.abs(z_test - z_back))):.2e}")

n_params = sum(p.size for p in jax.tree.leaves(nnx.state(flow, nnx.Param)))
print(f"Parameters in the flow: {n_params:,}")


The round-trip error should be tiny — limited by floating-point precision, not by the network. The flow is *literally* invertible by construction.

### A quick note on signs and directions

The `log_prob(x)` method has a step that often confuses people the first time. We want the **density at a data point $x$**. To compute it we must first send $x$ **backwards** through the flow to find $z = f^{-1}(x)$, and only then apply change of variables. The formula reads

$$
\log p_X(x) \;=\; \log p_Z(z) \;+\; \log\!\left|\det J_{f^{-1}}(x)\right|
\;=\; \log p_Z(z) \;-\; \log\!\left|\det J_{f}(z)\right|.
$$

Both forms are correct; they're related by $J_{f^{-1}} = J_f^{-1}$, so the log magnitudes flip sign. Our `inverse()` method returns $\log|\det J_{f^{-1}}|$ directly (that's the `log_det` it accumulates), which is why `log_prob` uses **`+ log_det_inv`** rather than `-`. If you ever get a sign wrong, trace which direction your `log_det` came from — that's almost always the culprit.


## 7. Hands-on B — train on a 2D shape

We'll fit two toy datasets: **two moons** and **two concentric rings**. Neither can be captured by a single Gaussian, so they're a good test of the flow.

The training objective is *maximum likelihood*: maximise the average $\log p_\theta(x)$ on data.

In code: `loss = -flow.log_prob(x).mean()`. That's it.


In [ ]:
def make_moons(n, noise=0.06, key=jr.PRNGKey(0)):
    k1, k2, k3 = jr.split(key, 3)
    n_half = n // 2
    theta_upper = jnp.linspace(0, jnp.pi, n_half)
    theta_lower = jnp.linspace(0, jnp.pi, n - n_half)
    upper = jnp.stack([jnp.cos(theta_upper), jnp.sin(theta_upper)], axis=1)
    lower = jnp.stack([1 - jnp.cos(theta_lower), 1 - jnp.sin(theta_lower) - 0.5], axis=1)
    data = jnp.concatenate([upper, lower], axis=0)
    data += noise * jr.normal(k2, data.shape)
    return data[jr.permutation(k3, len(data))]


def make_rings(n, key=jr.PRNGKey(0)):
    k1, k2, k3 = jr.split(key, 3)
    n1, n2 = n // 2, n - n // 2
    theta1 = jr.uniform(k1, (n1,), maxval=2*jnp.pi)
    theta2 = jr.uniform(k2, (n2,), maxval=2*jnp.pi)
    r1 = 1.0 + 0.08 * jr.normal(k2, (n1,))
    r2 = 2.0 + 0.08 * jr.normal(k3, (n2,))
    return jnp.concatenate([
        jnp.stack([r1*jnp.cos(theta1), r1*jnp.sin(theta1)], axis=1),
        jnp.stack([r2*jnp.cos(theta2), r2*jnp.sin(theta2)], axis=1),
    ], axis=0)


# Visualise the targets
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
moons = make_moons(2000); rings = make_rings(2000)
axes[0].scatter(moons[:, 0], moons[:, 1], s=4, alpha=0.5); axes[0].set_title("two moons")
axes[1].scatter(rings[:, 0], rings[:, 1], s=4, alpha=0.5); axes[1].set_title("two rings")
for ax in axes: ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
def train_flow(flow, data, n_epochs=300, batch_size=256, lr=3e-4, seed=42):
    optimizer = nnx.Optimizer(flow, optax.adam(lr), wrt=nnx.Param)

    def loss_fn(flow, x):
        return -flow.log_prob(x).mean()       # negative log-likelihood

    @nnx.jit
    def train_step(flow, optimizer, x):
        loss, grads = nnx.value_and_grad(loss_fn, argnums=nnx.DiffState(0, nnx.Param))(flow, x)
        optimizer.update(flow, grads)
        return loss

    losses = []
    key = jr.PRNGKey(seed)
    for epoch in range(n_epochs):
        key, sub = jr.split(key)
        perm = jr.permutation(sub, len(data))
        data_shuf = data[perm]
        ep_loss, nb = 0.0, 0
        for i in range(0, len(data), batch_size):
            xb = data_shuf[i:i+batch_size]
            if len(xb) < 2: continue
            ep_loss += float(train_step(flow, optimizer, xb)); nb += 1
        losses.append(ep_loss / max(nb, 1))
        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1:4d}   NLL = {losses[-1]:.4f}")
    return losses


In [ ]:
# Train on two moons
data_moons = make_moons(5000, key=jr.PRNGKey(10))
flow_moons = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(0))
loss_moons = train_flow(flow_moons, data_moons, n_epochs=300)


In [ ]:
# Train on rings
data_rings = make_rings(5000, key=jr.PRNGKey(20))
flow_rings = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(1))
loss_rings = train_flow(flow_rings, data_rings, n_epochs=300)


## 8. Did it work? — three diagnostics

We inspect the trained flow in three ways.

1. **Samples**: draw $z \sim \mathcal{N}(0, I)$, push through the flow, plot the result. Does it look like the data?
2. **Density**: evaluate $p_\theta(x)$ on a grid. Does it light up where the data is?
3. **Latent**: push the *data* backwards through the flow. Does it land on a Gaussian-looking blob?

If all three checks pass, the flow has done its job.


In [ ]:
def plot_flow_results(flow, data, title, ax_samples, ax_density, ax_latent,
                       xlim=(-3, 3), ylim=(-3, 3)):
    samples = np.asarray(flow.sample(jr.PRNGKey(99), 3000))
    ax_samples.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.4, c="C1")
    ax_samples.scatter(data[:500, 0], data[:500, 1], s=3, alpha=0.3, c="C0", label="data")
    ax_samples.set_title(f"{title} — samples vs data"); ax_samples.legend(fontsize=8, markerscale=3)
    ax_samples.set_aspect("equal"); ax_samples.grid(alpha=0.3)

    nx = 100
    xg = jnp.linspace(xlim[0], xlim[1], nx); yg = jnp.linspace(ylim[0], ylim[1], nx)
    xx, yy = jnp.meshgrid(xg, yg)
    grid = jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
    log_p = np.asarray(flow.log_prob(grid)).reshape(nx, nx)
    ax_density.contourf(np.asarray(xx), np.asarray(yy), np.exp(log_p), levels=30, cmap="viridis")
    ax_density.set_title(f"{title} — learned density"); ax_density.set_aspect("equal")

    z_data, _ = flow.inverse(jnp.asarray(data[:2000]))
    z_data = np.asarray(z_data)
    ax_latent.scatter(z_data[:, 0], z_data[:, 1], s=3, alpha=0.3, c="C2")
    theta = np.linspace(0, 2*np.pi, 100)
    for r in [1, 2]:
        ax_latent.plot(r*np.cos(theta), r*np.sin(theta), "k--", alpha=0.3, lw=0.8)
    ax_latent.set_title(f"{title} — data pushed back to latent space")
    ax_latent.set_aspect("equal"); ax_latent.set_xlim(-4, 4); ax_latent.set_ylim(-4, 4)
    ax_latent.grid(alpha=0.3)


fig, axes = plt.subplots(2, 3, figsize=(13, 8))
plot_flow_results(flow_moons, np.asarray(data_moons), "Moons",
                  axes[0,0], axes[0,1], axes[0,2], xlim=(-1.5, 2.5), ylim=(-1.0, 1.5))
plot_flow_results(flow_rings, np.asarray(data_rings), "Rings",
                  axes[1,0], axes[1,1], axes[1,2], xlim=(-3, 3), ylim=(-3, 3))
plt.tight_layout(); plt.show()


**What to look for.**

- Orange samples match the blue data — the flow generates plausible new examples.
- The density (middle column) should concentrate along the data shape. This is the **exact** $p_\theta(x)$, not a bound — so visible artifacts are model limitations, not estimator noise.
- The latent scatter (right column) should look closer to a Gaussian than the original data. The rings example is deliberately harder for this tiny RealNVP, so imperfect normalization is a useful diagnostic.


## 9. Watch the deformation happen — layer by layer

Let's look at how the flow actually builds up its transformation. We feed Gaussian noise in and snapshot the points after each coupling layer.


In [ ]:
key = jr.PRNGKey(42)
z0 = jr.normal(key, (1500, 2))

snapshots = [np.asarray(z0)]
h = z0
for layer in flow_moons.layers:
    h, _ = layer.forward(h)
    snapshots.append(np.asarray(h))

n_show = min(len(snapshots), 9)
idx = np.linspace(0, len(snapshots)-1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(2.8*n_show, 2.8))
for ax, i in zip(axes, idx):
    pts = snapshots[i]
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3)
    if i == 0:                       ax.set_title(r"$z \sim \mathcal{N}(0,I)$", fontsize=10)
    elif i == len(snapshots)-1:      ax.set_title(r"$x = f(z)$", fontsize=10)
    else:                            ax.set_title(f"after layer {i}", fontsize=10)
    ax.set_aspect("equal"); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
plt.suptitle("Gaussian → Two Moons: morphing one layer at a time", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


Each layer makes a small, simple move along one axis. **Stack a dozen and you get a highly nonlinear, expressive transformation.** That's the whole magic.


## 9.5 Hands-on C — train a flow on real data (handwritten digits)

The toy 2D shapes were great for pictures, but you might be wondering: *does any of this work on actual data?* Let's spend a few minutes on a small real-world demo using **scikit-learn's handwritten digits** — 1797 images, each 8×8, so the data live in **64 dimensions**.

To keep training fast and stable on a CPU we'll first reduce the data to **16 dimensions with PCA** (the top 16 principal components capture ~73% of the pixel-level variance). The flow then learns the distribution *inside* that 16-D subspace; at the end we reconstruct images by inverting the PCA. This is a standard preprocessing trick — it doesn't change anything about the flow itself.

> **Caveat up front.** The 8×8 sklearn digits are only ~73% reconstructable from 16 PCs and the resulting flow samples will look like *plausible digit-shaped blobs*, not crisp characters. State-of-the-art image generation today uses diffusion models on high-dimensional pixel data, which we'll meet in the next lecture. The point here is **to see the same flow code from the 2D example scale to a real dataset**.


In [ ]:
# Quietly install sklearn if it's not already there (Colab usually has it).
try:
    import sklearn  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_raw = digits.data.astype("float32")           # (1797, 64), values in [0, 16]
print(f"raw data shape: {X_raw.shape}, value range: [{X_raw.min():.0f}, {X_raw.max():.0f}]")

# Show a few real digits so we know what we're trying to generate.
fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img, label in zip(axes.flat, X_raw[:16], digits.target[:16]):
    ax.imshow(img.reshape(8, 8), cmap="gray_r"); ax.axis("off")
    ax.set_title(str(label), fontsize=9)
plt.suptitle("Real handwritten digits — our training data", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Shuffle once, then split 80/20 into train and test so we can honestly check generalisation.
rng = np.random.default_rng(0)
perm = rng.permutation(len(X_raw))
n_train = int(0.8 * len(X_raw))
train_raw, test_raw = X_raw[perm[:n_train]], X_raw[perm[n_train:]]

# Standardize and PCA: fit ONLY on the training data, then apply to test.
# Pipeline: standardize pixels -> PCA to 16 components (with whitening so each
# component has unit variance, which is much friendlier for the flow).
scaler = StandardScaler().fit(train_raw)
pca    = PCA(n_components=16, whiten=True, random_state=0).fit(scaler.transform(train_raw))

X_train_pca = pca.transform(scaler.transform(train_raw)).astype("float32")
X_test_pca  = pca.transform(scaler.transform(test_raw)).astype("float32")
print(f"train: {X_train_pca.shape},  test: {X_test_pca.shape}")
print(f"PCA retains {pca.explained_variance_ratio_.sum():.2%} of the variance with 16 components")
print(f"per-component std after whitening: {float(X_train_pca.std(0).mean()):.3f}  (target = 1.0)")

# Train a 16-D RealNVP on the whitened PCA latents.
flow_digits = RealNVP(d=16, n_layers=10, hidden=128, rngs=nnx.Rngs(0))
loss_digits = train_flow(flow_digits, jnp.asarray(X_train_pca),
                         n_epochs=400, batch_size=128, lr=5e-4)


In [ ]:
# Sample novel "digits" from the trained flow.
# Pipeline: sample in 16-D PCA space -> inverse PCA -> de-standardize -> reshape to 8x8.
fake_pca = np.asarray(flow_digits.sample(jr.PRNGKey(7), 16))
fake_std = pca.inverse_transform(fake_pca)
fake_img = scaler.inverse_transform(fake_std).reshape(-1, 8, 8)

fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img in zip(axes.flat, fake_img):
    ax.imshow(np.clip(img, 0, 16), cmap="gray_r"); ax.axis("off")
plt.suptitle("New digits generated by the flow (no training image was reused)", y=1.02)
plt.tight_layout(); plt.show()


**What you should see.** The generated images look *digit-like* — round loops, vertical strokes, slanted bars — even though no specific training image was memorised. Some will be clearly recognisable; others will look like ambiguous in-between glyphs ("a 3 that's halfway to an 8"). That's exactly what a good generative model does: live on the manifold of plausible digits, including new points the data never showed.

### Use the flow as an exact density evaluator

Because flows give **exact log-likelihoods**, we can score the held-out test set and use it as a "did the model actually learn the distribution, or just memorise the training set?" check. Lower NLL on held-out data = better density model. A big gap between training NLL and test NLL is a classic overfitting signature.


In [ ]:
# Honest train/test NLL comparison.
train_nll = -float(flow_digits.log_prob(jnp.asarray(X_train_pca)).mean())
test_nll  = -float(flow_digits.log_prob(jnp.asarray(X_test_pca)).mean())

# Per-dimension version makes the numbers comparable across different d.
d = X_train_pca.shape[1]
print(f"NLL on training set ({len(X_train_pca)} imgs):  {train_nll:8.2f}   (per-dim: {train_nll/d:.3f})")
print(f"NLL on held-out test ({len(X_test_pca)} imgs):  {test_nll:8.2f}   (per-dim: {test_nll/d:.3f})")
print(f"generalisation gap              :  {test_nll - train_nll:+.2f}")
print()
print("Interpretation: positive gap means test is harder than train — some overfitting.")
print("Try retraining with more layers, more hidden units, or fewer epochs and see which way the gap moves.")


**Take-away from this section.** The exact same RealNVP class — no architectural change beyond bumping `d` from 2 to 16 — fits a real-world dataset and produces visually convincing samples. Nothing about coupling layers is specific to physics or to toy 2D distributions; it's a generic density estimator that scales.


## 10. Why a physicist should care — the alanine dipeptide Ramachandran distribution

> **Data: real (computed).** This section uses a molecular-dynamics trajectory of **alanine dipeptide** (Ac-Ala-NHMe), the classic toy molecule of biophysics — small enough to fit in a Colab notebook, yet it already shows the multi-basin behaviour that makes sampling hard.

A floppy molecule like alanine dipeptide has many atoms, but its slow conformational change is captured by just **two backbone dihedral angles**, $\phi$ and $\psi$. Plotting the equilibrium distribution in the $(\phi, \psi)$ plane gives the famous **Ramachandran plot**. It is multi-modal: the molecule spends its time in a few well-separated **basins**, named after the secondary structures they correspond to:

- the **$\alpha$-helix / $\alpha_R$ basin** near $(\phi, \psi) \approx (-1.0, -0.7)$ rad,
- the **$\beta$-sheet / $C5$ basin** near $(\phi, \psi) \approx (-2.2, +2.3)$ rad,
- a smaller left-handed $\alpha_L$ basin near $(+1.0, +0.7)$ rad.

At temperature $T$ the equilibrium distribution over angles is, again, **Boltzmann**:

$$
p(\phi, \psi) \;\propto\; e^{-F(\phi, \psi) / k_B T},
$$

where $F(\phi, \psi)$ is the **free energy surface** (the potential of mean force) along these two collective coordinates. Just like the double well, the basins are separated by **free-energy barriers**, and ordinary MCMC / MD gets **trapped** in one basin for a long time before it hops to the next. A flow is a one-shot sampler — it deforms the whole Gaussian at once and never has to climb a barrier.

> **Physics analogy.** $(\phi, \psi)$ are *collective coordinates / order parameters*; $F(\phi, \psi)$ is the *free-energy landscape*; the barriers are exactly what makes the problem interesting. This is the same target we will revisit in **L16 (Flow Matching)** and **L17 (Diffusion)** — switching the physics target here propagates through the whole generative block.

<!-- lecture15-visual:start:boltzmann-generator -->
<!-- lecture15-visual:end:boltzmann-generator -->


### Step 1 — load the real trajectory and extract $(\phi, \psi)$

`mdtraj` ships a small built-in alanine dipeptide trajectory (a few thousand frames, ~1 MB). We compute the backbone dihedrals with `mdtraj.compute_phi` / `compute_psi` and stack them into a 2D dataset of angles in **radians**. If the bundled file is not available on your runtime, we fall back to a synthetic-but-realistic Ramachandran mixture so the rest of the notebook still runs.


In [ ]:
# --- Load alanine dipeptide and extract (phi, psi) backbone dihedrals --------
# Requires:  !pip install -q mdtraj   (run the install cell at the top first)
def load_alanine_phipsi():
    # Return an (N, 2) array of (phi, psi) angles in radians.
    import mdtraj as md
    from mdtraj.testing import get_fn          # bundled example files
    traj = md.load(get_fn("alanine-dipeptide.pdb"))        # topology + frames
    # compute_phi/compute_psi return (indices, angles[n_frames, n_dihedrals])
    _, phi = md.compute_phi(traj)
    _, psi = md.compute_psi(traj)
    return np.stack([phi[:, 0], psi[:, 0]], axis=-1).astype(np.float32)

try:
    phipsi = load_alanine_phipsi()
    DATA_SOURCE = "mdtraj alanine-dipeptide trajectory (real MD)"
except Exception as e:
    # Fallback: a 3-basin Ramachandran mixture in (phi, psi) so the notebook is
    # self-contained even without mdtraj / network. Same downstream code path.
    print(f"[mdtraj unavailable: {e}]\n -> using a synthetic Ramachandran mixture instead.")
    rng = np.random.default_rng(42)
    centers = np.array([[-1.0, -0.7], [-2.2, 2.3], [1.0, 0.7]])   # alphaR, beta, alphaL
    weights = np.array([0.55, 0.40, 0.05])
    counts  = rng.multinomial(20000, weights)
    blobs = [c + 0.30 * rng.standard_normal((n, 2)) for c, n in zip(centers, counts)]
    phipsi = np.concatenate(blobs, axis=0).astype(np.float32)
    DATA_SOURCE = "synthetic Ramachandran mixture (fallback)"

# Wrap angles into (-pi, pi] so the 2D domain is a clean torus chart.
phipsi = (phipsi + np.pi) % (2 * np.pi) - np.pi
print(f"data source : {DATA_SOURCE}")
print(f"phipsi shape: {phipsi.shape}   (N frames, [phi, psi])")
print(f"phi range   : [{phipsi[:,0].min():+.2f}, {phipsi[:,0].max():+.2f}] rad")
print(f"psi range   : [{phipsi[:,1].min():+.2f}, {phipsi[:,1].max():+.2f}] rad")


In [ ]:
# Ramachandran plot of the REAL data: where does the molecule actually live?
plt.figure(figsize=(5.2, 5))
plt.hist2d(phipsi[:, 0], phipsi[:, 1], bins=80,
           range=[[-np.pi, np.pi], [-np.pi, np.pi]], cmap="viridis")
plt.colorbar(label="counts")
# annotate the named basins
for (px, py, name) in [(-1.0, -0.7, r"$\alpha_R$"), (-2.2, 2.3, r"$\beta$"), (1.0, 0.7, r"$\alpha_L$")]:
    plt.plot(px, py, "w*", ms=13, mec="k")
    plt.annotate(name, (px, py), textcoords="offset points", xytext=(8, 6), color="w", fontsize=12)
plt.xlabel(r"$\phi$ (rad)"); plt.ylabel(r"$\psi$ (rad)")
plt.title("Ramachandran plot — alanine dipeptide (data)")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()


### Step 2 — turn the data into a free-energy surface $F(\phi, \psi)$

To run a **Boltzmann generator** we need an *energy function* we can evaluate at any point, not just samples. For a real molecule that would be the force field; here we reconstruct the **free energy** directly from the trajectory by inverting Boltzmann's relation,

$$
F(\phi, \psi) \;=\; -k_B T \,\log p_{\text{data}}(\phi, \psi) \;+\; \text{const},
$$

estimating $p_{\text{data}}$ with a smoothed 2D histogram. This $F$ is then differentiable-free but cheap to query on a grid — exactly what the reverse-KL loss and the MCMC baseline (Section 11) both need. We set $k_B T = 1$ in these reduced angle units (the absolute scale only shifts the loss by a constant).


In [ ]:
from scipy.ndimage import gaussian_filter

kT = 1.0
NB = 80
edges = np.linspace(-np.pi, np.pi, NB + 1)
centers = 0.5 * (edges[:-1] + edges[1:])
H, _, _ = np.histogram2d(phipsi[:, 0], phipsi[:, 1], bins=[edges, edges])
H = gaussian_filter(H, sigma=1.2) + 1e-3      # smooth + floor (no -inf)
p_data = H / H.sum()
F_grid = -kT * np.log(p_data)                 # free energy, up to a constant
F_grid -= F_grid.min()                        # shift so min(F) = 0

# Build a JAX energy callable U(x)/kT by bilinear lookup into F_grid.
F_jax = jnp.asarray(F_grid)
phi_c = jnp.asarray(centers)
def energy_phipsi(x, kT=kT):
    # Reduced energy U(x)/kT at angle pairs x[..., (phi, psi)] in radians.
    # map angle in (-pi, pi] -> fractional bin index in [0, NB-1]
    idx = (x + jnp.pi) / (2 * jnp.pi) * NB - 0.5
    i = jnp.clip(idx, 0.0, NB - 1.0001)
    i0 = jnp.floor(i).astype(jnp.int32); fr = i - i0
    iphi0, ipsi0 = i0[..., 0], i0[..., 1]
    fphi, fpsi   = fr[..., 0], fr[..., 1]
    def g(a, b):  # gather F at integer bins (a, b)
        return F_jax[jnp.clip(a, 0, NB-1), jnp.clip(b, 0, NB-1)]
    F = ( g(iphi0,   ipsi0  ) * (1-fphi) * (1-fpsi)
        + g(iphi0+1, ipsi0  ) * fphi     * (1-fpsi)
        + g(iphi0,   ipsi0+1) * (1-fphi) * fpsi
        + g(iphi0+1, ipsi0+1) * fphi     * fpsi )
    return F / kT

# Visualise: free-energy surface and the corresponding Boltzmann density.
xx, yy = np.meshgrid(centers, centers, indexing="ij")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
im0 = axes[0].contourf(xx, yy, F_grid, levels=30, cmap="magma_r")
axes[0].set_title(r"Free energy $F(\phi,\psi) = -k_BT\log p_{\rm data}$"); plt.colorbar(im0, ax=axes[0])
axes[1].contourf(xx, yy, p_data, levels=30, cmap="viridis")
axes[1].set_title(r"Boltzmann density $\propto e^{-F/k_BT}$")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlabel(r"$\phi$ (rad)"); ax.set_ylabel(r"$\psi$ (rad)")
plt.tight_layout(); plt.show()


### Step 3 — train a flow on the energy alone (a Boltzmann generator)

We **never feed the flow the data points**. We only give it the energy function $U = F/k_BT$ and ask: *push Gaussian noise through $f_\theta$ — does the output look Boltzmann-distributed under $U$?* The objective is the **reverse KL divergence** from the flow's density $q_\theta$ to the target $p \propto e^{-U}$, which (dropping a constant) is

$$
\mathcal{L}(\theta) \;=\; \mathbb{E}_{z \sim \mathcal{N}(0, I)}\!\left[\; \frac{U\big(f_\theta(z)\big)}{k_B T} \;-\; \log\big|\det J_{f_\theta}(z)\big|\;\right].
$$

The first term pulls samples toward **low free energy** (the basins); the second term is a **volume-expansion** reward that stops the flow from collapsing all its mass onto a single point. Their balance reproduces the Boltzmann distribution. Implement it below.


In [ ]:
flow_boltz = RealNVP(d=2, n_layers=12, hidden=64, rngs=nnx.Rngs(42))
optimizer_b = nnx.Optimizer(flow_boltz, optax.adam(3e-4), wrt=nnx.Param)

def boltzmann_loss(flow, z, kT):
    x, log_det = flow.forward(z)        # x = f(z),  log_det = log|det J_f(z)|
    energy = energy_phipsi(x, kT=kT)    # U(x)/kT  for each sample, shape (batch,)
    # === TODO 3 — reverse-KL Boltzmann loss ============================
    # Return the MEAN over the batch of:   U(x)/kT  -  log|det J_f(z)|
    #   * first term  (energy)  -> minimise energy  = sit in the basins
    #   * second term (log_det) -> reward volume expansion = don't collapse
    # Both `energy` and `log_det` already have shape (batch,); combine and .mean().
    # return ...                                          # <-- TODO
    return (energy - log_det).mean()                      # reference solution
    # ===================================================================

@nnx.jit
def train_step_b(flow, optimizer, z, kT):
    loss, grads = nnx.value_and_grad(boltzmann_loss, argnums=nnx.DiffState(0, nnx.Param))(flow, z, kT)
    optimizer.update(flow, grads)
    return loss

losses_b, key = [], jr.PRNGKey(0)
for epoch in range(500):
    key, sub = jr.split(key)
    z = jr.normal(sub, (512, 2))
    losses_b.append(float(train_step_b(flow_boltz, optimizer_b, z, kT)))
    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch+1:4d}   loss = {losses_b[-1]:.3f}")

plt.figure(figsize=(7, 3))
plt.plot(losses_b); plt.xlabel("epoch"); plt.ylabel("reverse-KL loss")
plt.title("Boltzmann generator for alanine dipeptide — trained from the energy alone")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
# Checkpoint: the loss should decrease over the 500 epochs. If TODO 3 is wrong
# (e.g. wrong sign on log_det) the loss diverges or refuses to drop.


In [ ]:
# Did it work? Overlay flow samples on the data Ramachandran plot.
flow_samples = np.asarray(flow_boltz.sample(jr.PRNGKey(99), 8000))
# keep samples inside the (-pi, pi] chart for a fair comparison
m = np.all(np.abs(flow_samples) <= np.pi, axis=1)
flow_in = flow_samples[m]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))

axes[0].hist2d(phipsi[:, 0], phipsi[:, 1], bins=80,
               range=[[-np.pi, np.pi], [-np.pi, np.pi]], cmap="viridis")
axes[0].set_title("Data (MD trajectory)")

axes[1].scatter(flow_in[:, 0], flow_in[:, 1], s=3, alpha=0.25, c="C1")
axes[1].set_title("Flow — one-shot samples")

axes[2].contourf(xx, yy, p_data, levels=30, cmap="viridis")
axes[2].set_title(r"Target Boltzmann $\propto e^{-F/k_BT}$")

for ax in axes:
    ax.set_xlim(-np.pi, np.pi); ax.set_ylim(-np.pi, np.pi); ax.set_aspect("equal")
    ax.set_xlabel(r"$\phi$ (rad)"); ax.set_ylabel(r"$\psi$ (rad)")
    for (px, py) in [(-1.0, -0.7), (-2.2, 2.3), (1.0, 0.7)]:
        ax.plot(px, py, "w*", ms=10, mec="k")
plt.tight_layout(); plt.show()


**What you should see.** The trained flow:

- populates the **$\alpha_R$ and $\beta$ basins** (and ideally a hint of $\alpha_L$) of the Ramachandran plot, matching the named minima of the real data;
- produces **independent, one-shot samples** — no Markov chain, no barrier-crossing wait;
- **used no training conformations** — only the free-energy surface $F(\phi,\psi)$.

This is exactly the idea behind **Boltzmann generators** (Noé *et al.*, *Science* 2019), where the very first published example was — alanine dipeptide. In production the same recipe samples protein conformations and lattice gauge-field configurations.

> **Honest caveat (reverse KL).** Training from the energy with reverse KL has a famous failure mode — **mode collapse**: the flow may over-populate the deepest basin and under-sample the shallower $\alpha_L$ one. If you see that, it is not a bug, it is the asymmetry of $\mathrm{KL}(q_\theta \,\|\, p)$ at work. Production Boltzmann generators fix it by mixing in the *forward* KL (maximum likelihood on data) and importance reweighting. See Section 13.


## 11. Why we can't just use MCMC

Could we have sampled the same Ramachandran distribution with a Metropolis chain instead? Let's see what happens when we start two chains in two **different basins** of the alanine free-energy surface.

<!-- lecture15-visual:start:mcmc-vs-flow -->
<!-- lecture15-visual:end:mcmc-vs-flow -->


In [ ]:
# Simple Metropolis-Hastings on the SAME free-energy surface, via jax.lax.scan.
@partial(jax.jit, static_argnames=("energy_fn", "n_steps", "step_size"))
def metropolis_mcmc(energy_fn, x0, kT, n_steps, step_size, key):
    def step(carry, k):
        x, n_acc = carry
        k1, k2 = jr.split(k)
        x_prop = x + step_size * jr.normal(k1, x.shape)
        dE = energy_fn(x_prop, kT) - energy_fn(x, kT)
        accept = jr.uniform(k2) < jnp.exp(-dE)
        x_new = jnp.where(accept, x_prop, x)
        return (x_new, n_acc + accept.astype(jnp.int32)), x_new
    keys = jr.split(key, n_steps)
    (_, n_acc), traj = jax.lax.scan(step, (x0, jnp.int32(0)), keys)
    return traj, n_acc

# Two chains: one started in the alpha_R basin, one in the beta basin.
alphaR = jnp.array([-1.0, -0.7]); beta = jnp.array([-2.2, 2.3])
n_steps, step_size = 12000, 0.05
traj_A, _ = metropolis_mcmc(energy_phipsi, alphaR, kT, n_steps, step_size, jr.PRNGKey(0))
traj_B, _ = metropolis_mcmc(energy_phipsi, beta,   kT, n_steps, step_size, jr.PRNGKey(1))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
axes[0].scatter(np.asarray(traj_A)[:, 0], np.asarray(traj_A)[:, 1], s=2, alpha=0.2, c="C3")
axes[0].set_title(r"MCMC — start in $\alpha_R$ basin")
axes[1].scatter(np.asarray(traj_B)[:, 0], np.asarray(traj_B)[:, 1], s=2, alpha=0.2, c="C4")
axes[1].set_title(r"MCMC — start in $\beta$ basin")
axes[2].scatter(flow_in[:, 0], flow_in[:, 1], s=2, alpha=0.2, c="C1")
axes[2].set_title("Flow — independent samples")
for ax in axes:
    ax.set_xlim(-np.pi, np.pi); ax.set_ylim(-np.pi, np.pi); ax.set_aspect("equal")
    ax.set_xlabel(r"$\phi$ (rad)"); ax.set_ylabel(r"$\psi$ (rad)")
    for (px, py) in [(-1.0, -0.7), (-2.2, 2.3), (1.0, 0.7)]:
        ax.plot(px, py, "k*", ms=9)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Trapped on classroom timescales.** With small local proposals, each MCMC chain mostly explores the **basin where it started** — the $\alpha_R$ chain stays in $\alpha_R$, the $\beta$ chain stays in $\beta$. Crossing the free-energy ridge between them needs a run of improbable uphill moves, suppressed by roughly $e^{-\Delta F / k_B T}$.

A real-world conformational barrier can be $\Delta F / k_B T \sim 10$ to $30$, putting the crossing time far beyond any practical MD/MCMC run. The Boltzmann generator avoids this local-walk bottleneck entirely: it transforms the **whole Gaussian** in one shot, so it places mass in **every** basin simultaneously — exactly the third panel.


## 12. Recap

| What | In words |
|---|---|
| **Goal** | Learn an invertible $f$ that deforms a Gaussian into the data. |
| **Density** | $\log p_\theta(x) = \log p_Z(z) + \log|\det J_{f^{-1}}(x)|$ — exact, not a bound. |
| **Hard part** | Building $f$ so $\det J$ is cheap. |
| **Trick** | Coupling layers — split the variables, transform one half conditionally on the other. The Jacobian becomes triangular and the determinant collapses to a sum. |
| **Training (with data)** | Maximum likelihood = minimise $-\log p_\theta(x)$ on samples. |
| **Training (without data)** | Use the energy: $\mathcal{L} = \mathbb{E}_z[U(f(z))/k_BT - \log|\det J_f|]$. |
| **Killer app for physics** | Sampling Boltzmann distributions that MCMC cannot reach. |

### What's the catch?

- The latent and the data have the **same dimension**. A flow cannot compress like a VAE does.
- Designing a flow with both **expressiveness** and a **cheap Jacobian** is an active area of research. We saw the simplest (coupling). Others (autoregressive flows, neural splines, continuous flows) trade these off differently.
- Image-scale flows (Glow) work, but state-of-the-art image/molecule generation is now **continuous-time** flows and diffusion.

> **Where this goes next.** In **L16 (Flow Matching)** we replace the stack of discrete coupling layers with a single continuous-time velocity field $dx/dt = v_\theta(x, t)$, trained *simulation-free* — and we will apply it to **the very same alanine dipeptide Boltzmann target** you just built. **L17 (Diffusion)** is a particular continuous flow trained with a denoising objective. So today's notebook is the discrete-time root of the whole generative-modelling tree we spend the next three weeks climbing.


## 13. Going further (optional reading)

What we built today is **RealNVP** — the simplest flow that is both expressive and easy to train. Most production architectures are small variations on the same coupling-layer idea. Here's the family tree, with one paragraph per relative.

### Other coupling-layer flows

- **NICE** (Dinh, Krueger & Bengio, 2014) — the original coupling layer, *before* the scaling part was added. It used only $x_B = z_B + t(z_A)$ (additive only, no $e^s$). Because $\det J = 1$ everywhere, NICE is **volume-preserving** — it can't squeeze probability into a small region, so it tacks on a learned diagonal rescaling at the end as a workaround. RealNVP fixes this by allowing the scale $e^{s}$.
- **Glow** (Kingma & Dhariwal, 2018) — RealNVP for **images that actually work**. Two new ingredients: (i) a *1×1 invertible convolution* between coupling layers, which lets the model mix information across channels (instead of using a fixed checkerboard split); (ii) *actnorm*, a per-channel affine layer that replaces batch norm and keeps the Jacobian tractable. Glow is what made flow-based image generation competitive with GANs around 2018.
- **Neural Spline Flows** (Durkan *et al.*, 2019) — replace the simple affine `z * exp(s) + t` inside each coupling layer with a *monotonic rational-quadratic spline*. Much more flexible per layer, so you need fewer layers. Today this is the default "good flow" for tabular and scientific data.

### Autoregressive flows (the other big family)

- **MAF — Masked Autoregressive Flow** (Papamakarios, Pavlakou & Murray, 2017). Instead of splitting variables into halves, transform dimension $i$ conditioned on *all earlier dimensions* $1, \ldots, i-1$. Density evaluation is fast (one forward pass with a masked MLP), but sampling is sequential — $O(d)$ passes for $d$ dimensions. Good when you mainly need likelihoods.
- **IAF — Inverse Autoregressive Flow** (Kingma *et al.*, 2016). The mirror image: sampling is one shot, density evaluation is sequential. Good when you mainly need samples. Often used inside a VAE as a richer posterior.

(RealNVP and Glow are the *coupling* compromise that gives both fast sampling and fast density evaluation, at the cost of less expressiveness per layer.)

### Continuous-time flows

- **Neural ODE / FFJORD** (Chen *et al.*, 2018; Grathwohl *et al.*, 2019) — instead of discrete layers, define $dx/dt = v_\theta(x, t)$ and integrate. The Jacobian-determinant calculation turns into a trace, which is much cheaper than a determinant. This is the conceptual bridge to **diffusion models** and to flow matching.
- **Forward pointer → L16 (Flow Matching).** The continuous-time idea above is expensive to *train* because you must integrate the ODE (and its trace) during every gradient step. **Flow Matching / Conditional Flow Matching** (Lipman *et al.*, 2023) makes it **simulation-free**: it regresses $v_\theta(x,t)$ directly onto a known target velocity, with no ODE solves in the loss. In L16 we apply CFM to **the same alanine dipeptide Boltzmann target** from Section 10 — so keep this notebook handy, the physics target carries straight over.

### Equivariant and physics-aware flows

- Bake physical symmetries (translation, rotation, permutation of identical particles, periodic boundaries) directly into the coupling networks so the flow respects them by construction. Essential for serious molecular and lattice applications — without symmetry, the model wastes capacity re-learning the same physics from every orientation.
- Production examples: Noé *et al.* (*Science* 2019) for protein conformations; Albergo–Kanwar–Shanahan (*PRD* 2019) for lattice QCD.

### A bit more math (if you want it)

- Training on data with $-\overline{\log p_\theta(x)}$ is equivalent to minimising the **forward KL** $\mathrm{KL}(p_{\text{data}} \,\|\, p_\theta)$.
- The Boltzmann-generator loss from Section 10 is the **reverse KL** $\mathrm{KL}(p_\theta \,\|\, p_{\text{Boltzmann}})$ — these two directions are *not* symmetric and the reverse direction has a famous failure mode (mode collapse: the flow can ignore some of the Ramachandran basins if it's not careful). Production systems combine both directions with importance reweighting.

### Forward pointer → L19 (Simulation-Based Inference)

So far the flow has modelled a *single* density $p_\theta(x)$. Make the flow **conditional** — let its $s, t$ networks also take an observation $x_{\text{obs}}$ as input — and it learns a family $p_\theta(\theta \mid x_{\text{obs}})$. That is a **neural posterior estimator**: a flow that, given data, returns the full Bayesian posterior over physical parameters. In **L19 (Simulation-Based Inference)** we reuse the *exact same RealNVP code from this notebook*, only feeding the conditioning variable into the coupling networks, to do likelihood-free inference for physics simulators. Today's coupling layer is the workhorse there too — the role becomes concrete in the fifth week of the course.

We covered just enough to make the picture honest. The slides under `slides/` walk through the variational-inference background (ELBO, reparameterization trick) if you want that.


## 14. References

**Foundational papers**

1. Dinh, Sohl-Dickstein & Bengio, *Density Estimation Using Real-Valued Non-Volume Preserving (Real-NVP) Transformations*, ICLR 2017. arXiv:1605.08803.
2. Kingma & Dhariwal, *Glow: Generative Flow with Invertible 1×1 Convolutions*, NeurIPS 2018. arXiv:1807.03039.
3. Rezende & Mohamed, *Variational Inference with Normalizing Flows*, ICML 2015. arXiv:1505.05770.

**Physics applications**

4. Noé, Olsson, Köhler & Wu, *Boltzmann Generators: Sampling Equilibrium States of Many-Body Systems with Deep Learning*, *Science* 365, eaaw1147 (2019).
5. Albergo, Kanwar & Shanahan, *Flow-based Generative Models for MCMC in Lattice Field Theory*, *Phys. Rev. D* 100, 034515 (2019).

**Reviews**

6. Papamakarios *et al.*, *Normalizing Flows for Probabilistic Modeling and Inference*, JMLR 22 (2021). arXiv:1912.02762.
7. Kobyzev, Prince & Brubaker, *Normalizing Flows: An Introduction and Review of Current Methods*, IEEE TPAMI 43, 3964 (2021).

---

*End of Lecture 15.*
